# Task 11: full-history fit and inference

This notebook is a read-only inspection surface for the production artifact. The actual pipeline lives in `pipeline.py`, `submission.py`, and `scripts/run_full_fit_inference.py`. It does not launch training or submit to Kaggle.

In [ ]:
import json
from pathlib import Path

import polars as pl

repo_root = Path.cwd().parent if Path.cwd().name == 'research' else Path.cwd()
artifact = repo_root / 'artifacts' / 'task11_full_fit_v1'
config = json.loads((repo_root / 'configs' / 'task11_full_fit_v1.json').read_text())
config['run_id'], config['mode'], config['catboost']['iterations']

## Frozen training contract

The final ranker is one fixed-budget pointwise CatBoost fit over a bounded pool sampled from `rolling_1`, `rolling_2`, `rolling_3`, and `canonical`. Full-history candidates have no future labels and are used only for final inference.

In [ ]:
feature_schema = json.loads((artifact / 'feature_schema.json').read_text())
sampling = json.loads((artifact / 'sampling_manifest.json').read_text())
training = json.loads((artifact / 'training_manifest.json').read_text())
{
    'feature_count': feature_schema['feature_count'],
    'folds': sampling['fold_order'],
    'training_rows': sampling['actual_rows'],
    'positive_rows': sampling['actual_positive_rows'],
    'tree_count': training['tree_count'],
    'early_stopping': training['early_stopping'],
}

## Prediction and submission diagnostics

In [ ]:
metrics = json.loads((artifact / 'metrics.json').read_text())
submission_validation = json.loads((artifact / 'submission_validation.json').read_text())
recommendations = pl.scan_parquet(artifact / 'internal_recommendations.parquet')
recommendations.select(
    users=pl.len(),
    min_items=pl.col('item_ids').list.len().min(),
    max_items=pl.col('item_ids').list.len().max(),
).collect(), metrics, submission_validation

`metrics.json` intentionally contains no hidden-label Precision@20 or candidate recall. Those values are unknowable for leaderboard target users.

In [ ]:
pl.read_parquet(artifact / 'internal_recommendations.parquet').head(10)